# Cloud-cover case: sampling × model R², SHAP time, distance to a reference explanation

Same layout as `examples/obs_gpp_sample_model_shap.ipynb`.

Y is ERA5 total cloud cover (`tcc`). X is ERA5 2 m temperature, VPD, downward short-wave radiation, and 500 hPa vertical velocity, already on the ERA5 1° grid for 1980–2014. Time is stamped to `YYYY-MM-01` and inner-joined. Dewpoint is not used as a predictor because VPD already combines temperature and dewpoint.

1. Load cubes and flatten valid cells.
2. Sampling methods and sample sizes.
3. Models.
4. Test-set R² for every (n, method, model).
5. SHAP wall time and dependence-curve distance to a **reference** explanation (`extra_trees` + `random` + largest n), using `compare_explanations`.

Do not execute this notebook until you are ready; the grid can take a while.

In [ ]:
from pathlib import Path
import sys
import time

root = Path.cwd().resolve()
if not (root / "src" / "rf_shap").exists() and (root.parent / "src" / "rf_shap").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
fastforest_root = Path("/Users/huangfeini/Documents/FastForest")
if fastforest_root.exists() and str(fastforest_root) not in sys.path:
    sys.path.insert(0, str(fastforest_root))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from rf_shap.cube.align import stamp_year_month
from rf_shap.cube.tabular import cube_to_table
from rf_shap.explain.evaluate import compare_explanations
from rf_shap.explain.shap_explain import compute_shap
from rf_shap.models.registry import available_models, build_model
from rf_shap.models.sampling import sample_train
from rf_shap.models.split import time_split
from rf_shap.predict.metrics import regression_metrics

print("models:", available_models())

## 1. Read inputs (X) and output (Y)

ERA5 monthly 1° files live in `esm_case_data/obs`. `valid_time` is renamed, `w` is squeezed off its pressure level, and all times are stamped to month start (`ssrd` is 06:00) so the join is exact.

In [ ]:
OBS = Path("/Users/huangfeini/Documents/esm_case_data/obs")
FILES = {
    "tcc": "era5_monthly_1deg_1980-2014_total_cloud_cover.nc",
    "t2m": "era5_monthly_1deg_1980-2014_2m_temperature.nc",
    "vpd": "era5_monthly_1deg_1980-2014_vapour_pressure_deficit.nc",
    "ssrd": "era5_monthly_1deg_1980-2014_surface_solar_radiation_downwards.nc",
    "w": "era5_monthly_1deg_1980-2014_vertical_velocity_500hPa.nc",
}
FEATURES = ["t2m", "vpd", "ssrd", "w"]
TARGET = "tcc"
SEED = 42

def open_cube(path, rename=None):
    ds = xr.open_dataset(path)
    mapping = {}
    for src, dst in (("valid_time", "time"), ("latitude", "lat"), ("longitude", "lon")):
        if src in ds.dims or src in ds.coords:
            mapping[src] = dst
    if mapping:
        ds = ds.rename(mapping)
    extra = [d for d in ds.dims if d not in ("time", "lat", "lon")]
    if extra:
        ds = ds.isel({d: 0 for d in extra}, drop=True)
    ds = stamp_year_month(ds)
    if rename:
        data_name = rename if rename in ds.data_vars else list(ds.data_vars)[0]
        ds = ds[[data_name]].rename({data_name: rename})
    keep = [v for v in ds.data_vars if set(ds[v].dims) >= {"time", "lat", "lon"}]
    return ds[keep]

cubes = {name: open_cube(OBS / fname, rename=name) for name, fname in FILES.items()}
x = xr.merge([cubes[name] for name in FEATURES], join="inner")
y = cubes[TARGET]
x, y = xr.align(x, y, join="inner")
print("X", dict(x.sizes), list(x.data_vars))
print("Y", dict(y.sizes), list(y.data_vars))
print("time", pd.to_datetime(x.time.values[0]), "→", pd.to_datetime(x.time.values[-1]))
display(x)
display(y)

In [ ]:
table = cube_to_table(x[FEATURES], y, y_name=TARGET)
print("valid cells", table.n_samples, "of", table.extras["n_raw"], "features", table.feature_names)
split = time_split(table, test_size=0.2)
train_pool, test = split.train, split.test
print("train pool", train_pool.n_samples, "test", test.n_samples, "cutoff", split.extras.get("cutoff"))
lat_pool = np.asarray(train_pool.index.get_level_values("lat"))
lon_pool = np.asarray(train_pool.index.get_level_values("lon"))
time_pool = np.asarray(train_pool.index.get_level_values("time"))

## 2. Sampling methods and sample sizes

Sampling only happens inside the chronological training pool. The test set is fixed.

In [ ]:
SAMPLE_SIZES = [200, 500]#, 1000, 2000
SAMPLE_METHODS = [
    "random",
    "stratified",
    "systematic",
    "area",
    "process",
    "fscs",
    "cluster_medoid",
    "spatial",
    "kennard_stone",
    "maximin",
    "clhs",
    "aclhs",
    "extreme",
    "active",
]

def draw_train(method, n_samples):
    t0 = time.perf_counter()
    idx = sample_train(
        train_pool.X,
        train_pool.y,
        method=method,
        n_samples=n_samples,
        seed=SEED,
        lat=lat_pool,
        lon=lon_pool,
        time=time_pool,
    )
    return train_pool.take(idx), float(time.perf_counter() - t0)

print("n_samples", SAMPLE_SIZES)
print("methods", SAMPLE_METHODS)

## 3. Models

Only backends that import successfully are kept. TreeSHAP-style explainers work on RF / ExtraTrees / XGBoost / LightGBM; `fastforest` is skipped for SHAP if it does not support TreeSHAP.

In [ ]:
MODEL_SPECS = [
    ("extra_trees", {"n_estimators": 80, "max_depth": 12, "n_jobs": -1}),
    ("rf", {"n_estimators": 80, "max_depth": 12, "n_jobs": -1}),
    ("hist_gbm", {"max_iter": 120, "max_depth": 8}),
    ("xgboost", {"n_estimators": 120, "max_depth": 8, "n_jobs": -1}),
    ("lightgbm", {"n_estimators": 120, "max_depth": 8, "n_jobs": -1}),
    ("lightgbm_rf", {"n_estimators": 120, "max_depth": 12, "n_jobs": -1}),
]
installed = set(available_models())
MODEL_SPECS = [(name, params) for name, params in MODEL_SPECS if name in installed]
print("using models", [name for name, _ in MODEL_SPECS])

def fit_score(model_name, params, train_table):
    model = build_model(model_name, params, seed=SEED)
    t0 = time.perf_counter()
    model.fit(train_table.X, train_table.y)
    fit_s = float(time.perf_counter() - t0)
    pred = model.predict(test.X)
    metrics = regression_metrics(test.y, pred)
    metrics["fit_s"] = fit_s
    metrics["n_train"] = int(len(train_table.y))
    return model, metrics

## 4. R² for sample size × sampling method × model

In [ ]:
skill_rows = []
fitted = {}  # (n, method, model) -> (model, train_table)
for n_samples in SAMPLE_SIZES:
    for method in SAMPLE_METHODS:
        try:
            train_table, sample_s = draw_train(method, n_samples)
        except Exception as exc:
            print("skip sample", method, n_samples, type(exc).__name__, exc)
            continue
        for model_name, params in MODEL_SPECS:
            model, metrics = fit_score(model_name, params, train_table)
            fitted[(n_samples, method, model_name)] = (model, train_table)
            skill_rows.append({
                "n_samples": n_samples,
                "method": method,
                "model": model_name,
                "sample_s": sample_s,
                **metrics,
            })
            print(n_samples, method, model_name, "R2", round(metrics["r2"], 3))

skill = pd.DataFrame(skill_rows)
display(skill[["n_samples", "method", "model", "n_train", "r2", "rmse", "ubrmse", "sample_s", "fit_s"]])
r2_table = skill.pivot_table(index=["method", "n_samples"], columns="model", values="r2")
display(r2_table)

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_SPECS), figsize=(4.2 * max(len(MODEL_SPECS), 1), 4.0), sharey=True, squeeze=False)
for ax, (model_name, _) in zip(axes[0], MODEL_SPECS):
    sub = skill[skill["model"] == model_name]
    for method, group in sub.groupby("method"):
        ax.plot(group["n_samples"], group["r2"], marker="o", label=method)
    ax.set_title(model_name)
    ax.set_xlabel("n_samples")
    ax.set_ylabel("test R²")
    ax.grid(True, alpha=0.3)
axes[0][-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle("cloud cover: test R² vs sample size")
fig.tight_layout()
plt.show()

## 5. SHAP time and distance to the reference explanation

Reference: `extra_trees` + `random` + largest `n_samples`. Every run explains the **same** test subset so `compare_explanations` / `dependence_curve_distance` is on the same X. Distance is the mean feature-wise dependence-SHAP curve distance to that reference.

In [ ]:
N_EXPLAIN = 400
SHAP_METHOD = "auto"
REF_MODEL = "extra_trees"
REF_METHOD = "random"
REF_N = max(SAMPLE_SIZES)
explain_table = test.take(np.arange(min(N_EXPLAIN, test.n_samples)))

def shap_on_test(model, background):
    t0 = time.perf_counter()
    result = compute_shap(
        model,
        explain_table,
        method=SHAP_METHOD,
        max_samples=None,
        background_size=80,
        background=background,
        seed=SEED,
    )
    elapsed = float(time.perf_counter() - t0)
    return result, elapsed

ref_key = (REF_N, REF_METHOD, REF_MODEL)
if ref_key not in fitted:
    raise KeyError(f"reference {ref_key} was not fitted; check MODEL_SPECS / SAMPLE_METHODS")
ref_model, ref_train = fitted[ref_key]
ref_shap, ref_time = shap_on_test(ref_model, ref_train.X)
x_ref = explain_table.X[ref_shap.sample_positions]
print("reference", ref_key, "explain_s", round(ref_time, 3), "method", ref_shap.method)

shap_rows = []
for (n_samples, method, model_name), (model, train_table) in fitted.items():
    try:
        shap_res, explain_s = shap_on_test(model, train_table.X)
        x_other = explain_table.X[shap_res.sample_positions]
        compared = compare_explanations(
            x_ref=x_ref,
            shap_ref=ref_shap.values,
            x_other=x_other,
            shap_other=shap_res.values,
            feature_names=explain_table.feature_names,
        )
        dist = compared["dependence_distance"]["distance"]
        slope = compared["slope_difference"]["difference"].abs()
        shap_rows.append({
            "n_samples": n_samples,
            "method": method,
            "model": model_name,
            "explain_s": explain_s,
            "explain_method": shap_res.method,
            "shap_distance": float(dist.mean()),
            "shap_distance_max": float(dist.max()),
            "slope_abs_diff": float(slope.mean()),
            "is_reference": (n_samples, method, model_name) == ref_key,
        })
        print(n_samples, method, model_name, "explain_s", round(explain_s, 3), "dist", round(float(dist.mean()), 4))
    except Exception as exc:
        print("skip shap", n_samples, method, model_name, type(exc).__name__, exc)

shap_skill = pd.DataFrame(shap_rows)
display(shap_skill)
time_table = shap_skill.pivot_table(index=["method", "n_samples"], columns="model", values="explain_s")
dist_table = shap_skill.pivot_table(index=["method", "n_samples"], columns="model", values="shap_distance")
print("SHAP time (s)")
display(time_table)
print("mean dependence-SHAP distance to reference")
display(dist_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for method, group in shap_skill.groupby("method"):
    mean_t = group.groupby("n_samples")["explain_s"].mean()
    mean_d = group.groupby("n_samples")["shap_distance"].mean()
    axes[0].plot(mean_t.index, mean_t.values, marker="o", label=method)
    axes[1].plot(mean_d.index, mean_d.values, marker="o", label=method)
axes[0].set_title("SHAP time vs n_samples (mean over models)")
axes[0].set_xlabel("n_samples")
axes[0].set_ylabel("explain_s")
axes[1].set_title("SHAP distance to reference vs n_samples")
axes[1].set_xlabel("n_samples")
axes[1].set_ylabel("mean dependence distance")
for ax in axes:
    ax.grid(True, alpha=0.3)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.suptitle("cloud cover")
fig.tight_layout()
plt.show()

combo = skill.merge(shap_skill, on=["n_samples", "method", "model"], how="left")
display(combo[["n_samples", "method", "model", "r2", "explain_s", "shap_distance", "explain_method"]])